COT-6405.   
Project.  
Pedro Perez

In [ ]:
import math
import matplotlib.pyplot as plt
import random
import pandas as pd
import numpy as np
import time

Utility functions

In [ ]:
# function to plot the points and the closest pair

def plot_points_and_closest_pair(P, index1, index2, title):
    x_coords = [p[0] for p in P]
    y_coords = [p[1] for p in P]
    
    plt.scatter(x_coords, y_coords, color='blue', label='Points')
    
    # Plot the closest pair
    plt.plot([P[index1][0], P[index2][0]], [P[index1][1], P[index2][1]], color='red', linewidth=2, label='Closest Pair')
    
    plt.xlabel('X Coordinate')
    plt.ylabel('Y Coordinate')
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
# function to generate random points

def generate_random_points(num_points, x_range=(0, 100), y_range=(0, 100),seed=None):
    if seed is not None:
        random.seed(seed)
    noduplicate_points = set()

    points = []
    e = 0
    while e < num_points:
         x = random.randrange(*x_range)
         y = random.randrange(*y_range)
         if (x, y) not in noduplicate_points:
             noduplicate_points.add((x, y))
             e += 1
    points = list(noduplicate_points)

    return points

Functions implementing the algorithms

In [ ]:

def BruteForceClosestPoint(P):
    # P is a list of n points, n ≥ 2, each point is a tuple (x, y)
    # Returns the 1-based indices of the closest pair of points

    n = len(P)
    print("Running Brute Force Algorithm on", n, "points...")
    d_min = float('inf')
    index1 = -1
    index2 = -1
    # the range(n) function starts at 0 and goes up to n-1, so we use n-1 to exclude the last element in the first loop
    for i in range(n - 1):   # i will go from 0 to n-2 => n-1 elements
        # the range function is exclusive of the upper bound, so we use n to include the last element
        for j in range(i + 1, n):  # j will go from i+1 to n-1
            # Calculate Euclidean distance
            d = math.sqrt((P[i][0] - P[j][0])**2 + (P[i][1] - P[j][1])**2)
            if d < d_min:
                d_min = d
                index1 = i
                index2 = j
    print(f"Closest pair found at indices: {index1} and {index2} with distance: {d_min}")
    return index1+1, index2+1 # 1-based index

In [ ]:
def DivideandConquerClosestPointRec(Px, Py):
    #print("Px:", Px)
    #print("Py:", Py)
    n = len(Px)
    #print("Number of points in this recursion:", n)
    # base case: use brute force for small number of points
    if n <= 3:
        d_min = float('inf')
        index1 = -1
        index2 = -1
        for i in range(n - 1):
            for j in range(i + 1, n):
                # Calculate Euclidean distance
                d = math.sqrt((P[Px[i]][0] - P[Px[j]][0])**2 + (P[Px[i]][1] - P[Px[j]][1])**2)
                if d < d_min:
                    d_min = d
                    index1 = Px[i]  # 0 based index for now
                    index2 = Px[j]  # 0 based index for now
        #print("Base case indices:", index1, index2)
        return index1, index2

    
    mid = math.ceil(n/2)
    #print("Mid index:", mid)
    
    # Divide points into two halves
    Qx = Px[:mid]  
    Rx = Px[mid:]
    
    # Maintain sorted order by y-coordinate for the halves
    Qy = [p for p in Py if p in Qx]
    Ry = [p for p in Py if p in Rx]
    #print("Qx:", Qx)
    #print("Rx:", Rx)
    #print("Qy:", Qy)
    #print("Ry:", Ry)
    
    # Recursively find closest pairs in both halves
    (left_pair_index1, left_pair_index2) = DivideandConquerClosestPointRec(Qx, Qy)
    (right_pair_index1, right_pair_index2) = DivideandConquerClosestPointRec(Rx, Ry)
    #print("Left pair indices:", left_pair_index1, left_pair_index2)
    #print("Right pair indices:", right_pair_index1, right_pair_index2)
    
    # Find the smaller distance between the two pairs
    d_left = math.sqrt((P[left_pair_index1][0] - P[left_pair_index2][0])**2 + 
                        (P[left_pair_index1][1] - P[left_pair_index2][1])**2)
    d_right = math.sqrt((P[right_pair_index1][0] - P[right_pair_index2][0])**2 + 
                        (P[right_pair_index1][1] - P[right_pair_index2][1])**2)
    d_min = min(d_left, d_right)

    xstar = P[Qx[-1]][0]  # The last point in Qx is the rightmost point in the left half

    # build line x = xstar
    L = xstar

    # Build S as the points in P that are within distance d_min of the line x = L
    S =  [p for p in Px if abs(P[p][0] - L) < d_min]
    Sy = [p for p in Py if abs(P[p][0] - L) < d_min]
    #print("Line x =", L)
    #print("Points in S:", S)
    #print("Points in Sy:", Sy)


    # Check Sy for closer pairs
    d_sy_min = d_min
    for i in range(len(Sy)):
        for j in range(i + 1, min(i + 15, len(Sy))):
            dx = P[Sy[i]][0] - P[Sy[j]][0]
            dy = P[Sy[i]][1] - P[Sy[j]][1]
            d_sy = math.sqrt(dx**2 + dy**2)
            if d_sy < d_sy_min:
                d_sy_min = d_sy
                sy_index1 = Sy[i]
                sy_index2 = Sy[j]

    if d_sy_min < d_min:
        return sy_index1, sy_index2
    elif d_left < d_right:
        return left_pair_index1, left_pair_index2
    else:
        return right_pair_index1, right_pair_index2



In [ ]:
def DivideandConquerClosestPoint(P):

    # P is a list of n points, n ≥ 2, each point is a tuple (x, y)
    # Returns the 1-based indices of the closest pair of points
 
    n= len(P)
    print("Running Divide and Conquer Algorithm on", n, "points...")
    # Create a list of with the indices of the points
    Pindexed = [i for i in range(len(P))]    


    # Sort points by x-coordinate
    Px = sorted(Pindexed, key=lambda p: P[p][0])
    # Sort points by y-coordinate
    Py = sorted(Pindexed, key=lambda p: P[p][1])
    
    
    index1, index2 =  DivideandConquerClosestPointRec(Px, Py)
    print(f"Closest pair found at indices: {index1} and {index2} ")

    return index1+1, index2+1 # 1-based index

    

Sample testing.   
Test # 1   
Odd number of points

In [ ]:
P=generate_random_points(11,(-20,20),(-20,20),seed=42)  # Generate 11 random points with a fixed seed for reproducibility
P

In [ ]:

index1, index2 = BruteForceClosestPoint(P)
print("Testing Brute Force:")
print(f"The closest pair of points are at indices: {index1} and {index2}")

In [ ]:
plot_points_and_closest_pair(P, index1-1, index2-1, "Closest Pair of Points - Brute Force")  # Example with the closest pair found by BruteForceClosestPoint

In [ ]:
index1, index2 = DivideandConquerClosestPoint(P)
print("Testing Divide and Conquer:")
print(f"The closest pair of points are at indices: {index1} and {index2}") 

In [ ]:
plot_points_and_closest_pair(P,index1-1, index2-1, "Closest Pair of Points - Divide and Conquer")  # Example with the closest pair found by DivideandConquerClosestPoint

Sample testing.   
Test # 2   
Even number of points

In [ ]:
P=generate_random_points(12,(-20,20),(-20,20),seed=50)  # Generate 12 random points with a fixed seed for reproducibility
P

In [ ]:

index1, index2 = BruteForceClosestPoint(P)
print("Testing Brute Force:")
print(f"The closest pair of points are at indices: {index1} and {index2}")

In [ ]:
plot_points_and_closest_pair(P, index1-1, index2-1, "Closest Pair of Points - Brute Force")  # Example with the closest pair found by BruteForceClosestPoint

In [ ]:
index1, index2 = DivideandConquerClosestPoint(P)
print("Testing Divide and Conquer:")
print(f"The closest pair of points are at indices: {index1} and {index2}") 

In [ ]:
plot_points_and_closest_pair(P,index1-1, index2-1, "Closest Pair of Points - Divide and Conquer")  # Example with the closest pair found by DivideandConquerClosestPoint

Project experiments

In [ ]:
# number of iterations
m = 10

In [ ]:
# sample sizes to test
#sample_sizes = [i * 10**4 for i in range(1, 2)]  #
sample_sizes = [i * 10**4 for i in range(1, 11)]  #
sample_sizes

In [ ]:
# seeds for reproducibility
# created a a matrix of seeds for each sample size and each iteration
np.random.seed(51)
seeds = np.random.randint(0, 1000, size=(len(sample_sizes), m))
seeds


In [ ]:
bfrunningtimes = [[0 for _ in range(m)] for _ in range(len(sample_sizes))]
bfrunningtimes

In [ ]:
bfrunningtimes[0][0]

In [ ]:
dcrunningtimes = [[0 for _ in range(m)] for _ in range(len(sample_sizes))]
dcrunningtimes

In [ ]:
for s in sample_sizes:
    print(f"Testing sample size: {s}")
    for i in range(m):
        print(f"Generating {s} random points for iteration:", i+1)
        P = generate_random_points(s, (-32000, 32000), (-32000, 32000), seed=int(seeds[sample_sizes.index(s)][i]))
        print("Random points generated. Running algorithms...")
        start_time = time.time()
        BruteForceClosestPoint(P)
        end_time = time.time()
        print(f"Brute Force Algorithm completed in {end_time - start_time} seconds. ")
        bfrunningtimes[sample_sizes.index(s)][i] = end_time - start_time
        start_time = time.time()
        DivideandConquerClosestPoint(P)
        end_time = time.time()
        print(f"Divide and Conquer Algorithm completed in {end_time - start_time} seconds. ")
        dcrunningtimes[sample_sizes.index(s)][i] = end_time - start_time

In [ ]:
bfrunningtimes

In [ ]:
dcrunningtimes